In [4]:
import geopandas as gpd
import pyogrio


In [5]:
BUILDINGS = "../data/raw/nz-building-outlines.gpkg"   # use your actual filename
FLOOD = "../data/processed/flood_plains_fixed.gpkg"

In [6]:
print(pyogrio.list_layers(BUILDINGS))
print(pyogrio.list_layers(FLOOD))

[['nz_building_outlines' 'MultiPolygon']]
[['flood_plains_fixed' 'MultiPolygon']]


In [12]:
SUBURBS = ["Mount Albert", "Mount Roskill", "Sandringham"]
where = "suburb_locality IN ('Mount Albert', 'Mount Roskill', 'Sandringham')"

buildings = gpd.read_file(BUILDINGS, layer="nz_building_outlines", where=where)
flood = gpd.read_file(FLOOD, layer="flood_plains_fixed")

print(buildings)
print(len(buildings), buildings.crs)
print(len(flood), flood.crs)

       building_id name      use suburb_locality town_city  \
0          4014403       Unknown    Mount Albert  Auckland   
1          4015329       Unknown    Mount Albert  Auckland   
2          4014401       Unknown    Mount Albert  Auckland   
3          4015337       Unknown    Mount Albert  Auckland   
4          4015343       Unknown    Mount Albert  Auckland   
...            ...  ...      ...             ...       ...   
22767      3951956       Unknown     Sandringham  Auckland   
22768      3951957       Unknown     Sandringham  Auckland   
22769      3952083       Unknown     Sandringham  Auckland   
22770      3951697       Unknown     Sandringham  Auckland   
22771      3951647       Unknown     Sandringham  Auckland   

      territorial_authority      capture_method capture_source_group  \
0                  Auckland  Feature Extraction    NZ Aerial Imagery   
1                  Auckland  Feature Extraction    NZ Aerial Imagery   
2                  Auckland  Feature Ex

In [9]:
print("Invalid:", (~flood.is_valid).sum())
print("Empty:", flood.is_empty.sum())

flood["geometry"] = flood.geometry.make_valid()
print("Invalid after fix:", (~flood.is_valid).sum())

Invalid: 0
Empty: 0
Invalid after fix: 0


In [10]:
joined = gpd.sjoin(buildings, flood[["geometry"]], how="inner", predicate="intersects")
exposed_idx = joined.index.unique()
buildings["exposed"] = buildings.index.isin(exposed_idx).astype(int)

print("Exposed:", buildings["exposed"].sum())

Exposed: 3284


In [11]:
print(buildings["use"].value_counts(dropna=False))   # hospitals check: does 'Hospital' appear at all?

critical = buildings[(buildings["exposed"] == 1) & (buildings["use"].isin(["School", "Hospital"]))]
schools = critical.groupby(["name", "use"]).size().sort_values(ascending=False)
print(len(critical), "buildings at", len(schools), "facilities")
print(schools)

suburbs = buildings.groupby("suburb_locality").agg(total=("exposed", "size"), exposed=("exposed", "sum"))
suburbs["pct_exposed"] = (100 * suburbs["exposed"] / suburbs["total"]).round(1)
print(suburbs.sort_values("pct_exposed", ascending=False))

use
Unknown        22505
School           260
Supermarket        7
Name: count, dtype: int64
40 buildings at 8 facilities
name                          use   
Mt Roskill Grammar            School    20
Waikowhai Intermediate        School     8
Hay Park School               School     3
Mt Albert Grammar School      School     3
Lynfield College              School     2
Marist School (Mount Albert)  School     2
Marist College                School     1
Mt Roskill Primary School     School     1
dtype: int64
                 total  exposed  pct_exposed
suburb_locality                             
Sandringham       4351     1218         28.0
Mount Roskill    10802     1253         11.6
Mount Albert      7619      813         10.7


In [15]:
buildings.to_file("../data/processed/buildings_with_exposure.gpkg", driver="GPKG")
suburbs.to_csv("../../outputs/exposure_by_suburb.csv")
schools.to_csv("../../outputs/schools_exposed.csv", header=["buildings"])